# Model Training and Evaluation

This notebook follows a clear flow:
1. Setup and data loading
2. Baseline model training and evaluation
3. Optional baseline submission export
4. XGBoost-priority optimization
5. Final optimized submission export

In [1]:
# 1) Setup: import models, metrics, and utilities
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge, Lasso
from xgboost import XGBRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split

/Users/nirlapidot/Documents/Project new With Copilot/Regression/venv/lib/python3.13/site-packages/xgboost/compat.py:105: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
# 2) Load processed training data and create stratified holdout split
X_loaded = np.load('../data/processed/X_processed.npy', allow_pickle=True)
y = np.load('../data/processed/y_log.npy', allow_pickle=True)

# If X_loaded is an object array containing a sparse matrix, extract it
if hasattr(X_loaded, 'shape') and X_loaded.shape == () and hasattr(X_loaded.item(), 'toarray'):
    X = X_loaded.item().toarray()  # Convert sparse to dense
else:
    X = X_loaded  # Already dense

# Stratify by target quantile bins to balance price ranges in holdout/CV
y_bins = pd.qcut(pd.Series(y), q=5, labels=False, duplicates='drop')

X_train, X_test, y_train, y_test, y_bins_train, y_bins_test = train_test_split(
    X,
    y,
    y_bins,
    test_size=0.2,
    random_state=42,
    stratify=y_bins
)

print(f'X_train shape: {X_train.shape} | X_test shape: {X_test.shape}')
print('Stratified split completed using target quantile bins.')

X_train shape: (1168, 206) | X_test shape: (292, 206)
Stratified split completed using target quantile bins.


In [3]:
# 3) Define baseline parameter grids and stratified cross-validation
rf_params = {'n_estimators': [100, 200], 'max_depth': [None, 10, 20]}
knn_params = {'n_neighbors': [3, 5, 7]}
xgb_params = {'n_estimators': [100, 200], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]}
ridge_params = {'alpha': [0.1, 1.0, 10.0, 50.0]}
lasso_params = {'alpha': [0.0005, 0.001, 0.005, 0.01]}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_splits = list(skf.split(X_train, y_bins_train))
print('Prepared stratified CV splits for all model comparisons.')

Prepared stratified CV splits for all model comparisons.


In [4]:
# 4) Train baseline models with shared stratified GridSearchCV protocol
rf = GridSearchCV(
    RandomForestRegressor(random_state=42),
    rf_params,
    cv=cv_splits,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
rf.fit(X_train, y_train)

knn = GridSearchCV(
    KNeighborsRegressor(),
    knn_params,
    cv=cv_splits,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
knn.fit(X_train, y_train)

xgb = GridSearchCV(
    XGBRegressor(random_state=42, objective='reg:squarederror', n_jobs=1, tree_method='hist'),
    xgb_params,
    cv=cv_splits,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
xgb.fit(X_train, y_train)

ridge = GridSearchCV(
    Ridge(random_state=42),
    ridge_params,
    cv=cv_splits,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
ridge.fit(X_train, y_train)

lasso = GridSearchCV(
    Lasso(random_state=42, max_iter=10000),
    lasso_params,
    cv=cv_splits,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
lasso.fit(X_train, y_train)

print('Baseline CV RMSE (log scale):')
print(f"RF: {-rf.best_score_:.5f}")
print(f"KNN: {-knn.best_score_:.5f}")
print(f"XGB: {-xgb.best_score_:.5f}")
print(f"Ridge: {-ridge.best_score_:.5f}")
print(f"Lasso: {-lasso.best_score_:.5f}")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for 

Baseline CV RMSE (log scale):
RF: 0.13889
KNN: 0.17577
XGB: 0.12421
Ridge: 0.11746
Lasso: 0.11745


In [5]:
# 5) Build baseline VotingRegressor with clone-safe XGBoost wrapper
from sklearn.base import BaseEstimator, RegressorMixin, clone

class SklearnCompatibleXGBRegressor(RegressorMixin, BaseEstimator):
    _estimator_type = 'regressor'

    def __init__(self, estimator=None):
        self.estimator = estimator

    def fit(self, X, y):
        base_estimator = self.estimator if self.estimator is not None else XGBRegressor(
            objective='reg:squarederror', random_state=42
        )
        self.model_ = clone(base_estimator)
        self.model_.fit(X, y)
        return self

    def predict(self, X):
        return self.model_.predict(X)

rf_model = clone(rf.best_estimator_)
knn_model = clone(knn.best_estimator_)
xgb_model = SklearnCompatibleXGBRegressor(estimator=xgb.best_estimator_)

voting = VotingRegressor([
    ('rf', rf_model),
    ('knn', knn_model),
    ('xgb', xgb_model)
])

voting.fit(X_train, y_train)
print('Baseline VotingRegressor fit succeeded with rf + knn + xgb.')

Baseline VotingRegressor fit succeeded with rf + knn + xgb.


In [6]:
# 6) Evaluate baseline VotingRegressor on holdout
y_pred_log = voting.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test)

mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

print(f'Baseline RMSE: {rmse:.2f}')
print(f'Baseline MAE: {mae:.2f}')
print(f'Baseline R2: {r2:.4f}')

Baseline RMSE: 22263.81
Baseline MAE: 14268.84
Baseline R2: 0.8893


## 7) Optional Baseline Submission Export

This section exports a baseline submission from the current `voting` model.
If you continue to optimization, run the final export section at the end instead.

In [7]:
# Optional: export baseline submission
import os
import pandas as pd

# Load processed test features
X_test_processed_loaded = np.load('../data/processed/X_test_processed.npy', allow_pickle=True)

# Handle sparse matrix stored as numpy object scalar
if hasattr(X_test_processed_loaded, 'shape') and X_test_processed_loaded.shape == () and hasattr(X_test_processed_loaded.item(), 'toarray'):
    X_test_processed = X_test_processed_loaded.item().toarray()
else:
    X_test_processed = X_test_processed_loaded

# Load IDs preserved from test.csv
test_ids = np.load('../data/processed/test_ids.npy', allow_pickle=True)

# Predict in log space and convert back to price scale
test_pred_log = voting.predict(X_test_processed)
test_pred_price = np.expm1(test_pred_log)

# Build submission schema
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': test_pred_price
})

# Save baseline submission file
os.makedirs('../data/submissions', exist_ok=True)
submission_path = '../data/submissions/submission_baseline.csv'
submission.to_csv(submission_path, index=False)

print(f'Baseline submission saved to: {submission_path}')
print(submission.head())

Baseline submission saved to: ../data/submissions/submission_baseline.csv
     Id      SalePrice
0  1461  124535.000773
1  1462  153398.951895
2  1463  186895.227349
3  1464  192939.834067
4  1465  186929.997968


## 8) Optimization Round: Expanded Search + XGBoost-Priority Voting

This section strengthens model selection in two ways:
1. Expanded hyperparameter search for Random Forest, KNN, and XGBoost.
2. CV-driven XGBoost-priority weighted voting.

At the end of this cell, `voting` is set to the best holdout performer.

In [8]:
# Expanded tuning with stratified CV and XGBoost-priority optimization
import os
from sklearn.base import clone, BaseEstimator, RegressorMixin

# --- XGBoost wrapper for sklearn VotingRegressor compatibility ---
class SklearnCompatibleXGBRegressor(RegressorMixin, BaseEstimator):
    _estimator_type = 'regressor'
    def __init__(self, estimator=None):
        self.estimator = estimator
    def fit(self, X, y):
        base_estimator = self.estimator if self.estimator is not None else XGBRegressor(objective='reg:squarederror', random_state=42)
        self.model_ = clone(base_estimator)
        self.model_.fit(X, y)
        return self
    def predict(self, X):
        return self.model_.predict(X)

def wrap_xgb_for_voting(xgb_estimator):
    """
    Wraps an XGBRegressor (from GridSearchCV or otherwise) in the SklearnCompatibleXGBRegressor
    for use in VotingRegressor. This avoids ValueError: The estimator XGBRegressor should be a regressor.
    """
    return SklearnCompatibleXGBRegressor(estimator=clone(xgb_estimator))

rf_params_opt = {
    'n_estimators': [200, 400],
    'max_depth': [None, 12, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

knn_params_opt = {
    'n_neighbors': [5, 7, 11, 15],
    'weights': ['uniform', 'distance'],
    'p': [1, 2]
}

xgb_params_opt = {
    'n_estimators': [300, 500],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.03, 0.05],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.7, 0.9],
    'min_child_weight': [1, 3],
    'gamma': [0.0, 0.1],
    'reg_alpha': [0.01, 0.1],
    'reg_lambda': [1.0, 3.0]
}

ridge_params_opt = {'alpha': [0.5, 1.0, 5.0, 10.0, 25.0]}
lasso_params_opt = {'alpha': [0.0005, 0.001, 0.005, 0.01]}

rf_opt = GridSearchCV(
    RandomForestRegressor(random_state=42),
    rf_params_opt,
    cv=cv_splits,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
rf_opt.fit(X_train, y_train)

knn_opt = GridSearchCV(
    KNeighborsRegressor(),
    knn_params_opt,
    cv=cv_splits,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
knn_opt.fit(X_train, y_train)

xgb_opt = GridSearchCV(
    XGBRegressor(
        random_state=42,
        objective='reg:squarederror',
        n_jobs=1,
        tree_method='hist'
    ),
    xgb_params_opt,
    cv=cv_splits,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
xgb_opt.fit(X_train, y_train)

ridge_opt = GridSearchCV(
    Ridge(random_state=42),
    ridge_params_opt,
    cv=cv_splits,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
ridge_opt.fit(X_train, y_train)

lasso_opt = GridSearchCV(
    Lasso(random_state=42, max_iter=10000),
    lasso_params_opt,
    cv=cv_splits,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
lasso_opt.fit(X_train, y_train)

single_model_summary = pd.DataFrame([
    {'model': 'RandomForest', 'cv_rmse_log': -rf_opt.best_score_, 'best_params': rf_opt.best_params_},
    {'model': 'KNN', 'cv_rmse_log': -knn_opt.best_score_, 'best_params': knn_opt.best_params_},
    {'model': 'XGBoost', 'cv_rmse_log': -xgb_opt.best_score_, 'best_params': xgb_opt.best_params_},
    {'model': 'Ridge', 'cv_rmse_log': -ridge_opt.best_score_, 'best_params': ridge_opt.best_params_},
    {'model': 'Lasso', 'cv_rmse_log': -lasso_opt.best_score_, 'best_params': lasso_opt.best_params_}
]).sort_values('cv_rmse_log', ascending=True)

print('Tuned single-model CV scores (lower is better):')
print(single_model_summary[['model', 'cv_rmse_log']])

# Use the strongest linear model as an ensemble candidate
if ridge_opt.best_score_ >= lasso_opt.best_score_:
    linear_name = 'Ridge'
    linear_best_estimator = clone(ridge_opt.best_estimator_)
    linear_cv_rmse = -ridge_opt.best_score_
else:
    linear_name = 'Lasso'
    linear_best_estimator = clone(lasso_opt.best_estimator_)
    linear_cv_rmse = -lasso_opt.best_score_

rf_best = clone(rf_opt.best_estimator_)
knn_best = clone(knn_opt.best_estimator_)
xgb_best = wrap_xgb_for_voting(xgb_opt.best_estimator_)
lin_best = clone(linear_best_estimator)

# XGB-priority voting search with a light linear contribution
weight_grid = [
    (0.15, 0.05, 0.70, 0.10),
    (0.10, 0.05, 0.75, 0.10),
    (0.10, 0.10, 0.70, 0.10),
    (0.10, 0.05, 0.80, 0.05),
    (0.20, 0.05, 0.65, 0.10)
]

best_weighted_model = None
best_weighted_cv = np.inf
best_weights = None

for w_rf, w_knn, w_xgb, w_lin in weight_grid:
    candidate = VotingRegressor(
        estimators=[
            ('rf', clone(rf_best)),
            ('knn', clone(knn_best)),
            ('xgb', wrap_xgb_for_voting(xgb_opt.best_estimator_)),
            ('lin', clone(lin_best))
        ],
        weights=[w_rf, w_knn, w_xgb, w_lin]
    )

    fold_rmse = []
    for train_idx, val_idx in cv_splits:
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        candidate.fit(X_tr, y_tr)
        y_val_pred_log = candidate.predict(X_val)
        fold_rmse.append(np.sqrt(mean_squared_error(y_val, y_val_pred_log)))

    mean_cv_rmse_log = float(np.mean(fold_rmse))

    if mean_cv_rmse_log < best_weighted_cv:
        best_weighted_cv = mean_cv_rmse_log
        best_weights = (w_rf, w_knn, w_xgb, w_lin)
        best_weighted_model = candidate

# Refit selected models on full training split
best_weighted_model.fit(X_train, y_train)
xgb_direct = wrap_xgb_for_voting(xgb_opt.best_estimator_)
xgb_direct.fit(X_train, y_train)
linear_direct = clone(linear_best_estimator)
linear_direct.fit(X_train, y_train)

# Holdout metrics in original price scale
y_true_opt = np.expm1(y_test)

y_pred_log_weighted = best_weighted_model.predict(X_test)
y_pred_weighted = np.expm1(y_pred_log_weighted)
rmse_weighted = np.sqrt(mean_squared_error(y_true_opt, y_pred_weighted))
mae_weighted = mean_absolute_error(y_true_opt, y_pred_weighted)
r2_weighted = r2_score(y_true_opt, y_pred_weighted)

y_pred_log_xgb = xgb_direct.predict(X_test)
y_pred_xgb = np.expm1(y_pred_log_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_true_opt, y_pred_xgb))
mae_xgb = mean_absolute_error(y_true_opt, y_pred_xgb)
r2_xgb = r2_score(y_true_opt, y_pred_xgb)

y_pred_log_lin = linear_direct.predict(X_test)
y_pred_lin = np.expm1(y_pred_log_lin)
rmse_lin = np.sqrt(mean_squared_error(y_true_opt, y_pred_lin))
mae_lin = mean_absolute_error(y_true_opt, y_pred_lin)
r2_lin = r2_score(y_true_opt, y_pred_lin)

print('\nBest weighted voting CV RMSE (log space):', round(best_weighted_cv, 5))
print('Best voting weights (rf, knn, xgb, lin):', best_weights)
print(f'Best linear model: {linear_name} | CV RMSE(log): {linear_cv_rmse:.5f}')

print('\nHoldout metrics on price scale (Weighted Voting):')
print(f'RMSE: {rmse_weighted:.2f}')
print(f'MAE: {mae_weighted:.2f}')
print(f'R2: {r2_weighted:.4f}')

print('\nHoldout metrics on price scale (Pure XGBoost):')
print(f'RMSE: {rmse_xgb:.2f}')
print(f'MAE: {mae_xgb:.2f}')
print(f'R2: {r2_xgb:.4f}')

print(f'\nHoldout metrics on price scale ({linear_name}):')
print(f'RMSE: {rmse_lin:.2f}')
print(f'MAE: {mae_lin:.2f}')
print(f'R2: {r2_lin:.4f}')

# Promote best holdout model
holdout_candidates = {
    'XGB-priority Weighted Voting': (rmse_weighted, best_weighted_model),
    'Pure XGBoost': (rmse_xgb, xgb_direct),
    linear_name: (rmse_lin, linear_direct)
}
chosen_model_name, (_, voting) = min(holdout_candidates.items(), key=lambda x: x[1][0])
print(f'\nChosen model for submission: {chosen_model_name}')

# Residual diagnostics by price quartiles
residuals = y_true_opt - np.expm1(voting.predict(X_test))
price_bins = pd.qcut(pd.Series(y_true_opt), q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'], duplicates='drop')
residual_summary = pd.DataFrame({'residual': residuals, 'price_bin': price_bins})
print('\nResidual mean by price quartile:')
print(residual_summary.groupby('price_bin', observed=False)['residual'].mean())

# Experiment log for reproducibility
experiment_log = pd.DataFrame([
    {'model': 'RandomForest_opt', 'cv_rmse_log': -rf_opt.best_score_, 'holdout_rmse': np.nan, 'best_params': str(rf_opt.best_params_)},
    {'model': 'KNN_opt', 'cv_rmse_log': -knn_opt.best_score_, 'holdout_rmse': np.nan, 'best_params': str(knn_opt.best_params_)},
    {'model': 'XGBoost_opt', 'cv_rmse_log': -xgb_opt.best_score_, 'holdout_rmse': rmse_xgb, 'best_params': str(xgb_opt.best_params_)},
    {'model': f'{linear_name}_opt', 'cv_rmse_log': linear_cv_rmse, 'holdout_rmse': rmse_lin, 'best_params': str(linear_best_estimator.get_params())},
    {'model': 'WeightedVoting_opt', 'cv_rmse_log': best_weighted_cv, 'holdout_rmse': rmse_weighted, 'best_params': str({'weights': best_weights})},
    {'model': 'ChosenFinal', 'cv_rmse_log': np.nan, 'holdout_rmse': min(rmse_weighted, rmse_xgb, rmse_lin), 'best_params': chosen_model_name}
])

os.makedirs('../data/submissions', exist_ok=True)
experiment_log_path = '../data/submissions/experiment_log.csv'
experiment_log.to_csv(experiment_log_path, index=False)
print(f'Experiment log saved to: {experiment_log_path}')


Fitting 5 folds for each of 24 candidates, totalling 120 fits
Fitting 5 folds for each of 16 candidates, totalling 80 fits
Fitting 5 folds for each of 768 candidates, totalling 3840 fits
Fitting 5 folds for each of 5 candidates, totalling 25 fits
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Tuned single-model CV scores (lower is better):
          model  cv_rmse_log
3         Ridge     0.117374
4         Lasso     0.117450
2       XGBoost     0.118575
0  RandomForest     0.138683
1           KNN     0.164933

Best weighted voting CV RMSE (log space): 0.11624
Best voting weights (rf, knn, xgb, lin): (0.1, 0.05, 0.75, 0.1)
Best linear model: Ridge | CV RMSE(log): 0.11737

Holdout metrics on price scale (Weighted Voting):
RMSE: 20295.78
MAE: 13416.14
R2: 0.9080

Holdout metrics on price scale (Pure XGBoost):
RMSE: 19962.68
MAE: 13812.07
R2: 0.9110

Holdout metrics on price scale (Ridge):
RMSE: 44350.31
MAE: 14889.90
R2: 0.5606

Chosen model for submission: Pure XGBoost

Res

## 9) Final Submission Export (After Optimization)

Run this section after the optimization cell. It exports predictions from the final selected `voting` model to the main submission path.

In [9]:
# Final export using the optimized selected model
import pandas as pd

X_test_processed_loaded = np.load('../data/processed/X_test_processed.npy', allow_pickle=True)

if hasattr(X_test_processed_loaded, 'shape') and X_test_processed_loaded.shape == () and hasattr(X_test_processed_loaded.item(), 'toarray'):
    X_test_processed = X_test_processed_loaded.item().toarray()
else:
    X_test_processed = X_test_processed_loaded

test_ids = np.load('../data/processed/test_ids.npy', allow_pickle=True)

test_pred_log = voting.predict(X_test_processed)
test_pred_price = np.expm1(test_pred_log)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': test_pred_price
})

os.makedirs('../data/submissions', exist_ok=True)
submission_path = '../data/submissions/submission.csv'
submission.to_csv(submission_path, index=False)

print(f'Final submission saved to: {submission_path}')
if 'chosen_model_name' in globals():
    print(f'Model used: {chosen_model_name}')
print(submission.head())

Final submission saved to: ../data/submissions/submission.csv
Model used: Pure XGBoost
     Id      SalePrice
0  1461  119241.250000
1  1462  162280.921875
2  1463  178688.640625
3  1464  188552.421875
4  1465  194793.390625
